**Training of Computer Vision Model**

In [ ]:
!pip install ultralytics

In [ ]:
from pathlib import Path

root = Path("../../data/raw_data/final_unified_dataset")
print("Dataset exists:", root.exists())
print("Train images:", (root / "images/train").exists())
print("Val images:", (root / "images/val").exists())
print("Test images:", (root / "images/test").exists())
print("Train labels:", (root / "labels/train").exists())
print("Val labels:", (root / "labels/val").exists())
print("Test labels:", (root / "labels/test").exists())
print("YAML exists:", (root / "dataset.yaml").exists())
print(Path("../../data/raw_data/final_unified_dataset/dataset.yaml").read_text())

**Experimental Models**

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    device = "mps"
)
metrics = model.val()
print(metrics)

In [ ]:
model = YOLO("yolov8s.pt") 

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    device="mps",
    workers=4,
    patience=20,
    project="yolov8s"
)

In [ ]:
model = YOLO("yolov8n.pt") 

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=10,
    device="mps",
    workers=2,
    patience=20,
    project="yolov8n"
)


In [ ]:
#Checking for class distribution
import os
from collections import Counter

label_dir = "../../data/raw_data/final_unified_dataset/labels/train"
class_counts = Counter()

for fname in os.listdir(label_dir):
    if not fname.endswith(".txt"):
        continue
    with open(os.path.join(label_dir, fname)) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

print("Class distribution in val labels:")
for cls_id, count in sorted(class_counts.items()):
    print(f"  Class {cls_id}: {count} instances")

In [ ]:
#Trying out a newer model

from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,              
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo11n",
)

*Re-running YOLOv11 on the updated dataset with more people*

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,              
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo11n_Updated_Dataset",
    cache =True
)

In [ ]:
import os
from collections import Counter

base = "../../data/raw_data/final_unified_dataset"

for split in ["train", "val", "test"]:
    label_dir = os.path.join(base, "labels", split)
    counter = Counter()
    for file in os.listdir(label_dir):
        if file.endswith(".txt"):
            with open(os.path.join(label_dir, file)) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        counter[int(parts[0])] += 1
    print(f"{split}: person={counter[0]}, bag={counter[1]}, ratio={counter[1]/counter[0]:.1f}x")

In [ ]:
import os

base = "../../data/raw_data/final_unified_datasetv1"

for split in ["train", "val", "test"]:
    label_dir = os.path.join(base, "labels", split)
    
    person_only = bag_only = both = empty = 0
    
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        classes = set()
        with open(os.path.join(label_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    classes.add(int(parts[0]))
        
        if not classes:
            empty += 1
        elif classes == {0}:
            person_only += 1
        elif classes == {1}:
            bag_only += 1
        elif 0 in classes and 1 in classes:
            both += 1
    
    print(f"\n{split}:")
    print(f"  Person only: {person_only}")
    print(f"  Bag only:    {bag_only}")
    print(f"  Both:        {both}")
    print(f"  Empty:       {empty}")

In [ ]:
from ultralytics import YOLO

test_image = "../../data/raw_data/final_unified_datasetv1/images/train/0d574e297bf42ad9.jpg"

# Your trained model
your_model = YOLO("../../runs/detect/yolo11n/train/weights/best.pt")

# Pretrained COCO
coco_model = YOLO("yolov8n.pt")

# Your model — person detections only
your_results = your_model.predict(test_image, classes=[0], conf=0.25, verbose=False)

# COCO model — person detections only  
coco_results = coco_model.predict(test_image, classes=[0], conf=0.25, verbose=False)

print(f"Your model   — people found: {len(your_results[0].boxes)}")
print(f"COCO model   — people found: {len(coco_results[0].boxes)}")

from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

def draw_boxes(image_path, results, title):
    img = Image.open(image_path).copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls  = int(box.cls[0])
        color = "cyan" if cls == 0 else "orange"
        label = f"{'person' if cls==0 else 'bag'} {conf:.2f}"
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1, y1-15), label, fill=color)
    
    return img

# Run both models showing ALL detections
your_all  = your_model.predict(test_image, conf=0.25, verbose=False)
coco_all  = coco_model.predict(test_image, conf=0.25, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(draw_boxes(test_image, your_all,  "Your model"))
axes[0].set_title("Your model (person + bag)")
axes[0].axis("off")
axes[1].imshow(draw_boxes(test_image, coco_all,  "COCO pretrained"))
axes[1].set_title("COCO pretrained (person only)")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("model_comparison_visual.png", dpi=100)
plt.show()

In [ ]:
import os

label_dir = "../../data/raw_data/final_unified_dataset/labels/train"
label_path = os.path.join(label_dir, "0d574e297bf42ad9.txt")

if os.path.exists(label_path):
    with open(label_path) as f:
        lines = f.readlines()
    
    print(f"Found: {label_path}")
    print(f"Number of annotations: {len(lines)}")
    print()
    
    class_names = {0: "person", 1: "bag"}
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if parts:
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            print(f"Annotation {i+1}:")
            print(f"  Class:  {cls} ({class_names.get(cls, 'unknown')})")
            print(f"  Center: ({cx:.4f}, {cy:.4f})")
            print(f"  Size:   {bw:.4f} x {bh:.4f}")
else:
    print(f"Not found in train labels: {label_path}")
    
    # Search all splits
    base = "../../data/raw_data/final_unified_dataset/labels"
    for split in ["train", "val", "test"]:
        path = os.path.join(base, split, "0d574e297bf42ad9.txt")
        if os.path.exists(path):
            print(f"Found in {split}: {path}")

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Randomly sample and visualise person-labelled images
img_dir   = "../../data/raw_data/final_unified_dataset/images/train"
label_dir = "../../data/raw_data/final_unified_dataset/labels/train"

# Find images that have person labels (class 0)
person_images = []
for fname in os.listdir(label_dir):
    if not fname.endswith(".txt"):
        continue
    with open(os.path.join(label_dir, fname)) as f:
        for line in f:
            if line.startswith("0 "):
                person_images.append(fname.replace(".txt", ".jpg"))
                break

print(f"Found {len(person_images)} images with person labels")

# Plot 9 random samples
sample = random.sample(person_images, min(9, len(person_images)))
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for ax, fname in zip(axes.flat, sample):
    img_path = os.path.join(img_dir, fname)
    lbl_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))

    if not os.path.exists(img_path):
        continue

    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)

    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])

            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            bw_px = bw * w
            bh_px = bh * h

            color = "cyan" if cls == 0 else "orange"
            label = "person" if cls == 0 else "bag"

            rect = patches.Rectangle(
                (x1, y1), bw_px, bh_px,
                linewidth=2, edgecolor=color, facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(x1, y1-5, label, color=color, fontsize=8)

    ax.set_title(fname[:30])
    ax.axis("off")

plt.tight_layout()
plt.savefig("label_verification.png", dpi=100)
plt.show()

Trying the combination of pe-built person detection and the old dataset's bag detection

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("../../runs/detect/yolo11n_Updated_Dataset/train/weights/best.pt")

yolo_metrics = yolo_model.val(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    split="val",
    verbose=False,
)

yolo_results = {
    "mAP50":        yolo_metrics.box.map50,
    "mAP50-95":     yolo_metrics.box.map,
    "precision":    yolo_metrics.box.mp,
    "recall":       yolo_metrics.box.mr,
    "person_mAP50": yolo_metrics.box.ap50[0],
    "bag_mAP50":    yolo_metrics.box.ap50[1],
    "inference_ms": yolo_metrics.speed["inference"],
}

print("YOLO11n results:")
for k, v in yolo_results.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
import os
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
import time

# ── Dataset ──────────────────────────────────────────────────────────────────

class YOLOFormatDataset(Dataset):
    """Loads YOLO format labels and converts to Faster R-CNN format"""

    def __init__(self, img_dir, label_dir):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        self.imgs = [
            f for f in os.listdir(img_dir)
            if f.endswith((".jpg", ".png"))
        ]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        fname    = self.imgs[idx]
        img_path = os.path.join(self.img_dir, fname)
        lbl_path = os.path.join(
            self.label_dir,
            fname.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        # Convert to tensor
        img_tensor = torchvision.transforms.functional.to_tensor(img)

        boxes  = []
        labels = []

        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    cls, cx, cy, bw, bh = map(float, parts)

                    # YOLO normalised → absolute pixel coordinates
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    x2 = (cx + bw/2) * w
                    y2 = (cy + bh/2) * h

                    # Clamp to image boundaries
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(w, x2), min(h, y2)

                    if x2 > x1 and y2 > y1:
                        boxes.append([x1, y1, x2, y2])
                        labels.append(int(cls) + 1)  # R-CNN uses 1-indexed, 0=background

        if boxes:
            boxes_tensor  = torch.tensor(boxes,  dtype=torch.float32)
            labels_tensor = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes_tensor  = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,),   dtype=torch.int64)

        target = {
            "boxes":    boxes_tensor,
            "labels":   labels_tensor,
            "image_id": torch.tensor([idx]),
        }

        return img_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


# ── Model ─────────────────────────────────────────────────────────────────────

def build_model(num_classes):
    """
    num_classes = your classes + 1 background
    So for person + bag: num_classes = 3
    """
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


# ── Evaluation ────────────────────────────────────────────────────────────────

def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1,y1,x2,y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0


def compute_ap(precisions, recalls):
    """Compute Average Precision using 11-point interpolation"""
    ap = 0
    for t in np.arange(0, 1.1, 0.1):
        prec = [p for p, r in zip(precisions, recalls) if r >= t]
        ap += max(prec) if prec else 0
    return ap / 11


def evaluate(model, dataloader, device, iou_threshold=0.5):
    model.eval()

    # class_id → list of (confidence, is_tp)
    all_detections = defaultdict(list)
    all_gt_counts  = defaultdict(int)

    with torch.no_grad():
        for images, targets in dataloader:
            images = [img.to(device) for img in images]
            preds  = model(images)

            for pred, target in zip(preds, targets):
                gt_boxes  = target["boxes"].cpu().numpy()
                gt_labels = target["labels"].cpu().numpy()

                pred_boxes  = pred["boxes"].cpu().numpy()
                pred_labels = pred["labels"].cpu().numpy()
                pred_scores = pred["scores"].cpu().numpy()

                # Count ground truth per class
                for lbl in gt_labels:
                    all_gt_counts[lbl] += 1

                # Match predictions to ground truth
                matched_gt = set()
                for box, lbl, score in sorted(
                    zip(pred_boxes, pred_labels, pred_scores),
                    key=lambda x: -x[2]
                ):
                    gt_indices = np.where(gt_labels == lbl)[0]
                    best_iou, best_idx = 0, -1

                    for gi in gt_indices:
                        iou = compute_iou(box, gt_boxes[gi])
                        if iou > best_iou:
                            best_iou = iou
                            best_idx = gi

                    is_tp = (
                        best_iou >= iou_threshold and
                        best_idx not in matched_gt
                    )

                    if is_tp:
                        matched_gt.add(best_idx)

                    all_detections[lbl].append((score, is_tp))

    # Compute AP per class
    class_names = {1: "person", 2: "bag"}
    aps = {}

    for cls_id, dets in all_detections.items():
        dets.sort(key=lambda x: -x[0])
        tp_cumsum = np.cumsum([d[1] for d in dets])
        total     = len(dets)
        gt_count  = all_gt_counts[cls_id]

        precisions = tp_cumsum / (np.arange(total) + 1)
        recalls    = tp_cumsum / gt_count if gt_count > 0 else np.zeros(total)

        ap = compute_ap(precisions.tolist(), recalls.tolist())
        aps[cls_id] = ap
        name = class_names.get(cls_id, f"class_{cls_id}")
        print(f"  {name}: AP50={ap:.4f}")

    map50 = np.mean(list(aps.values())) if aps else 0
    print(f"  mAP50: {map50:.4f}")
    return map50, aps


# ── Training loop ─────────────────────────────────────────────────────────────

def train(num_epochs=20):
    base = "../../data/raw_data/final_unified_dataset"

    # Use the old dataset — same one as your best yolo11n run
    train_dataset = YOLOFormatDataset(
        img_dir   = os.path.join(base, "images", "train"),
        label_dir = os.path.join(base, "labels", "train"),
    )
    val_dataset = YOLOFormatDataset(
        img_dir   = os.path.join(base, "images", "val"),
        label_dir = os.path.join(base, "labels", "val"),
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size  = 4,       # R-CNN needs more memory, keep batch small
        shuffle     = True,
        collate_fn  = collate_fn,
        num_workers = 0,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size  = 4,
        shuffle     = False,
        collate_fn  = collate_fn,
        num_workers = 0,
    )

    print(f"Train: {len(train_dataset)} images")
    print(f"Val:   {len(val_dataset)} images")

    # Device — R-CNN doesn't support MPS fully, use CPU on Mac
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Device: {device}")

    # num_classes = person + bag + background = 3
    model = build_model(num_classes=3)
    model.to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.005,
        momentum=0.9,
        weight_decay=0.0005,
    )
    lr_scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=5, gamma=0.5
    )

    history = {
        "epoch": [], "train_loss": [], "mAP50": [],
        "person_ap": [], "bag_ap": []
    }

    best_map  = 0
    best_path = "fasterrcnn_best.pt"
    start     = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        n_batches  = 0

        for images, targets in train_loader:
            images  = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses    = sum(loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            epoch_loss += losses.item()
            n_batches  += 1

        lr_scheduler.step()
        avg_loss = epoch_loss / n_batches

        print(f"\nEpoch {epoch+1}/{num_epochs} — Loss: {avg_loss:.4f}")
        map50, aps = evaluate(model, val_loader, device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(avg_loss)
        history["mAP50"].append(map50)
        history["person_ap"].append(aps.get(1, 0))
        history["bag_ap"].append(aps.get(2, 0))

        if map50 > best_map:
            best_map = map50
            torch.save(model.state_dict(), best_path)
            print(f"  Saved best model (mAP50={best_map:.4f})")

    duration = (time.time() - start) / 60
    print(f"\nTraining complete in {duration:.1f} minutes")
    print(f"Best mAP50: {best_map:.4f}")

    return history


# ── Plot results ──────────────────────────────────────────────────────────────

def plot_results(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Faster R-CNN training results", fontsize=13)

    axes[0].plot(history["epoch"], history["train_loss"], color="steelblue")
    axes[0].set_title("Training loss")
    axes[0].set_xlabel("Epoch")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["epoch"], history["mAP50"], color="purple", label="mAP50")
    axes[1].set_title("mAP50")
    axes[1].set_xlabel("Epoch")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(history["epoch"], history["person_ap"], label="person", color="teal")
    axes[2].plot(history["epoch"], history["bag_ap"],    label="bag",    color="orange")
    axes[2].set_title("Per-class AP50")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("fasterrcnn_results.png", dpi=150)
    plt.show()


# ── Run ───────────────────────────────────────────────────────────────────────

history = train(num_epochs=20)
plot_results(history)

In [ ]:
import os

base = "../../data/raw_data/final_unified_dataset"

for split in ["train", "val", "test"]:
    label_dir = os.path.join(base, "labels", split)
    person_only = bag_only = both = 0
    
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        classes = set()
        with open(os.path.join(label_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    classes.add(int(parts[0]))
        
        if classes == {0}:
            person_only += 1
        elif classes == {1}:
            bag_only += 1
        elif 0 in classes and 1 in classes:
            both += 1
    
    print(f"{split}: person_only={person_only}, bag_only={bag_only}, both={both}")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,              
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo11n_Dataset_v4",
    cache =True
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,              
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo8n_Dataset_v4",
    cache =True    
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov10n.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo10n_Dataset_v4",
    cache=True,
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=30,              
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo8m_Dataset_v4",
    cache =True    
)

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

label_dir = "../../data/raw_data/final_unified_dataset/labels/train"
img_dir   = "../../data/raw_data/final_unified_dataset/images/train"

# Find images with person labels
person_files = []
for fname in os.listdir(label_dir):
    if not fname.endswith(".txt"):
        continue
    with open(os.path.join(label_dir, fname)) as f:
        for line in f:
            if line.startswith("0 "):
                person_files.append(fname)
                break

print(f"Images with person labels: {len(person_files)}")

# Visualise 6 random ones
sample = random.sample(person_files, 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, fname in zip(axes.flat, sample):
    img_path = os.path.join(img_dir, fname.replace(".txt", ".jpg"))
    if not os.path.exists(img_path):
        continue
    
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)
    
    with open(os.path.join(label_dir, fname)) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            color = "cyan" if cls == 0 else "orange"
            label = "person" if cls == 0 else "bag"
            rect = patches.Rectangle(
                (x1, y1), bw*w, bh*h,
                linewidth=2, edgecolor=color, facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(x1, y1-5, label, color=color,
                   fontsize=9, backgroundcolor="black")
    
    ax.set_title(fname[:25])
    ax.axis("off")

plt.tight_layout()
plt.savefig("person_label_check.png", dpi=100)
plt.show()
EExplE

In [ ]:
import os

# Search for last.pt anywhere in your project
for root, dirs, files in os.walk(r"C:\Users\HP Victus\CDS_2026Spring_project"):
    for file in files:
        if file in ("last.pt", "best.pt"):
            print(os.path.join(root, file))

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import psutil
print(f"RAM available: {psutil.virtual_memory().available / 1e9:.1f} GB")
print(f"RAM total: {psutil.virtual_memory().total / 1e9:.1f} GB")

### MAIN TRAINING SCRIPT FOR SEAN

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11m.pt")

results = model.train(
    data=r"C:\Users\HP Victus\CDS_2026Spring_project\final_unified_dataset\dataset.yaml",
    
    # ── Training duration ──────────────────────────────────────────
    epochs=150,          # was 100 — bag recall needs more time to converge
    patience=40,         # was 30 — give it more room before early stopping

    # ── Batch / hardware ──────────────────────────────────────────
    imgsz=640,           # keeping original — safe for your GPU
    batch=16,
    device="0",
    workers=4,
    amp=True,
    seed=0,

    # ── Class imbalance fix ───────────────────────────────────────
    cls=1.5,             # upweight classification loss — penalises
                         # missing the minority bag class more strongly

    # ── Box localisation (mAP50-95 gap) ──────────────────────────
    box=9.0,             # tighter box regression — addresses the
                         # ~20pp mAP50→mAP50-95 drop

    # ── NMS (duplicate box fix) ───────────────────────────────────
    iou=0.4,             # stricter NMS suppresses the duplicate
                         # overlapping boxes seen in the video

    # ── Augmentation — compensates for no resolution bump ─────────
    scale=0.6,           # wider scale range catches small/distant objects
    copy_paste=0.3,      # bumped from 0.2 — pastes bag instances onto new
                         # backgrounds, directly boosts effective bag samples
    mixup=0.15,          # blends two images — exposes model to bags
                         # overlapping people in unusual ways
    degrees=10.0,        # mild rotation — bags carried at many angles
    fliplr=0.5,          # horizontal flip (default, explicitly kept)
    mosaic=1.0,          # keep mosaic on (default, explicitly kept)
    close_mosaic=15,     # was 10 — keep mosaic longer before turning it off

    # ── Output ────────────────────────────────────────────────────
    project=r"C:\Users\HP Victus\CDS_2026Spring_project\runs\detect\yolo11m_COCO_V2.1",
    name="exp2",
    plots=True,
)

print(f"Training complete! Results saved to: {results.save_dir}")

In [ ]:
from ultralytics import YOLO
import numpy as np

model = YOLO("../../runs/detect/yolo11m_COCO_V2/exp2/weights/best.pt")
metrics = model.val(
    data="../../final_unified_dataset/dataset.yaml",
    imgsz=640,
    split='test',
    plots=True,
)

print(f"Person AP50: {metrics.box.ap50[0]:.3f}")
print(f"Bag    AP50: {metrics.box.ap50[1]:.3f}")
print(f"Person AP50-95: {metrics.box.ap[0]:.3f}")
print(f"Bag    AP50-95: {metrics.box.ap[1]:.3f}")

In [ ]:
import os
from pathlib import Path

label_dir = Path(r"C:\Users\HP Victus\CDS_2026Spring_project\final_unified_dataset\labels\train")

bag_sizes = []
tiny_count = 0
total_bags = 0

for lbl_file in label_dir.glob("*.txt"):
    with open(lbl_file) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
            if cls == 1:  # bag
                total_bags += 1
                area = w * h
                bag_sizes.append(area)
                if area < 0.005:
                    tiny_count += 1

bag_sizes.sort()
print(f"Total bag annotations: {total_bags}")
print(f"Tiny bags (area<0.005): {tiny_count} ({100*tiny_count/total_bags:.1f}%)")
print(f"Median bag area: {bag_sizes[len(bag_sizes)//2]:.4f}")
print(f"Bottom 25% area: {bag_sizes[len(bag_sizes)//4]:.4f}")
print(f"Bottom 10% area: {bag_sizes[len(bag_sizes)//10]:.4f}")

In [ ]:
# Run prediction on a single image or a folder of images
results = model.predict(source=r"C:\Users\HP Victus\CDS_2026Spring_project\final_unified_dataset\test\images", save=True, conf=0.5)



In [ ]:
from ultralytics import YOLO
import cv2

# Load your best weights
model = YOLO("../../runs/detect/yolo11m_COCO_V3/exp_fast2/weights/best.pt")

# 1. Lower the confidence to 0.1 (10%) to see 'weak' detections
# 2. Set imgsz to 640 (or whatever you used in training)
results = model.predict(source="0", show=True, conf=0.5, imgsz=640)

In [8]:
from ultralytics import YOLO
import cv2
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
WEIGHT_PATH = "../../runs/detect/yolo11m_COCO_V2/exp2/weights/best.pt" # path to your trained weights
VIDEO_PATH  = "./test3.mp4"       # path to your test footage
OUTPUT_PATH = "./output3.mp4" # where to save the result
CONF        = 0.5
IMGSZ       = 640

# ── Load model ────────────────────────────────────────────────────────────────
model  = YOLO(WEIGHT_PATH)
cap    = cv2.VideoCapture(VIDEO_PATH)

# ── Match output video properties to input ────────────────────────────────────
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

print(f"Processing {total} frames at {fps:.1f} fps...")

# ── Run inference frame by frame ──────────────────────────────────────────────
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results    = model.predict(source=frame, conf=CONF, verbose=False,imgsz=IMGSZ,iou=0.3)
    annotated  = results[0].plot()

    out.write(annotated)

    # Progress every 50 frames
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"  {frame_idx}/{total} frames done ({frame_idx/total*100:.1f}%)")

cap.release()
out.release()
print(f"\n✅ Saved annotated video → {OUTPUT_PATH}")

Processing 626 frames at 30.0 fps...
  50/626 frames done (8.0%)
  100/626 frames done (16.0%)
  150/626 frames done (24.0%)
  200/626 frames done (31.9%)
  250/626 frames done (39.9%)
  300/626 frames done (47.9%)
  350/626 frames done (55.9%)
  400/626 frames done (63.9%)
  450/626 frames done (71.9%)
  500/626 frames done (79.9%)
  550/626 frames done (87.9%)
  600/626 frames done (95.8%)

✅ Saved annotated video → ./output3.mp4


In [ ]:
from pathlib import Path

base = r"C:\Users\HP Victus\CDS_2026Spring_project\new_dataset"
lbl_dir = Path(f"{base}/labels/train")

person_instances = 0
bag_instances    = 0

for f in lbl_dir.glob("*.txt"):
    for line in f.read_text().strip().split("\n"):
        if line.startswith("0"):
            person_instances += 1
        elif line.startswith("1"):
            bag_instances += 1

print(f"Person instances : {person_instances}")
print(f"Bag instances    : {bag_instances}")
print(f"Ratio            : {person_instances/max(bag_instances,1):.2f}x more persons than bags")

In [ ]:
from ultralytics import YOLO

model = YOLO(r"C:\Users\HP Victus\CDS_2026Spring_project\runs\detect\yolo11m_NewDataset\exp12\weights\best.pt")

for conf in [0.05, 0.10, 0.15, 0.25]:
    metrics = model.val(
        data=r"C:\Users\HP Victus\CDS_2026Spring_project\new_dataset\dataset.yaml",
        split="test",
        conf=conf,
        iou=0.5,
        plots=False,
    )
    print(f"conf={conf} → P:{metrics.box.mp:.3f} R:{metrics.box.mr:.3f} mAP50:{metrics.box.map50:.3f}")

#### Training only Bags model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    project="yolov8m_Bag_only_v5",
    plots=True,
    classes=[1],    # only train on bag (class 1)
)

##### Combining it with in-built dataset

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import os
import torch
from PIL import Image
import torchvision.ops as ops

bag_model  = YOLO("../../runs/detect/yolov8m_Bag_only_v5/train/weights/best.pt")
coco_model = YOLO("yolov8n.pt")

val_img_dir   = "../../data/raw_data/final_unified_dataset/images/val"
val_label_dir = "../../data/raw_data/final_unified_dataset/labels/val"

# ── IoU helper ────────────────────────────────────────────────────────────────

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0]);  y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]);  y2 = min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# ── Load ground truth ─────────────────────────────────────────────────────────

def load_ground_truth(label_path, img_w, img_h):
    """Load YOLO format labels and convert to absolute pixel coords"""
    gt_boxes  = []
    gt_labels = []
    if not os.path.exists(label_path):
        return gt_boxes, gt_labels
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls, cx, cy, bw, bh = map(float, parts)
            x1 = (cx - bw/2) * img_w
            y1 = (cy - bh/2) * img_h
            x2 = (cx + bw/2) * img_w
            y2 = (cy + bh/2) * img_h
            gt_boxes.append([x1, y1, x2, y2])
            gt_labels.append(int(cls))
    return gt_boxes, gt_labels

# ── Evaluate ──────────────────────────────────────────────────────────────────

def compute_ap(detections, n_gt):
    """Compute AP50 from list of (conf, is_tp) tuples"""
    if n_gt == 0:
        return 0.0
    detections.sort(key=lambda x: -x[0])
    tp_cumsum  = np.cumsum([d[1] for d in detections])
    precisions = tp_cumsum / (np.arange(len(detections)) + 1)
    recalls    = tp_cumsum / n_gt
    ap = 0
    for t in np.arange(0, 1.1, 0.1):
        prec = [p for p, r in zip(precisions, recalls) if r >= t]
        ap  += max(prec) if prec else 0
    return ap / 11

# Store per-class detections
all_detections = defaultdict(list)   # cls → [(conf, is_tp)]
all_gt_counts  = defaultdict(int)    # cls → total gt count

img_files = [f for f in os.listdir(val_img_dir) if f.endswith((".jpg", ".png"))]
print(f"Evaluating on {len(img_files)} val images...")

for i, fname in enumerate(img_files):
    if i % 50 == 0:
        print(f"  {i}/{len(img_files)}")

    img_path   = os.path.join(val_img_dir, fname)
    label_path = os.path.join(val_label_dir, fname.replace(".jpg", ".txt").replace(".png", ".txt"))

    img    = Image.open(img_path)
    img_w, img_h = img.size

    gt_boxes, gt_labels = load_ground_truth(label_path, img_w, img_h)

    # Count ground truth per class
    for lbl in gt_labels:
        all_gt_counts[lbl] += 1

    # Run both models
    bag_results  = bag_model.predict(img_path,  classes=[1], conf=0.29, verbose=False)
    coco_results = coco_model.predict(img_path, classes=[0], conf=0.40, verbose=False)

    # Collect predictions — bags from bag_model, people from coco_model
    predictions = []  # [(box, cls, conf)]

    for box in bag_results[0].boxes:
        predictions.append((
            box.xyxy[0].cpu().tolist(),
            1,                          # bag
            float(box.conf[0])
        ))

    for box in coco_results[0].boxes:
        predictions.append((
            box.xyxy[0].cpu().tolist(),
            0,                          # person
            float(box.conf[0])
        ))

    # Match predictions to ground truth
    matched_gt = set()
    for pred_box, pred_cls, pred_conf in sorted(predictions, key=lambda x: -x[2]):
        gt_indices = [i for i, l in enumerate(gt_labels) if l == pred_cls]
        best_iou, best_i = 0, -1

        for gi in gt_indices:
            iou = compute_iou(pred_box, gt_boxes[gi])
            if iou > best_iou:
                best_iou, best_i = iou, gi

        is_tp = best_iou >= 0.5 and best_i not in matched_gt
        if is_tp:
            matched_gt.add(best_i)

        all_detections[pred_cls].append((pred_conf, is_tp))

# ── Results ───────────────────────────────────────────────────────────────────

class_names = {0: "person", 1: "bag"}
aps = {}

print("\n" + "="*50)
print("COMBINED MODEL EVALUATION ON VAL SET")
print("="*50)

for cls_id in sorted(all_detections.keys()):
    dets   = all_detections[cls_id]
    n_gt   = all_gt_counts[cls_id]
    ap     = compute_ap(dets, n_gt)
    aps[cls_id] = ap

    dets_sorted = sorted(dets, key=lambda x: -x[0])
    tp_cumsum   = np.cumsum([d[1] for d in dets_sorted])
    n_dets      = len(dets_sorted)

    precision = tp_cumsum[-1] / n_dets if n_dets > 0 else 0
    recall    = tp_cumsum[-1] / n_gt   if n_gt   > 0 else 0

    name = class_names.get(cls_id, str(cls_id))
    source = "bag_model" if cls_id == 1 else "COCO"
    print(f"\n{name.upper()} (from {source})")
    print(f"  AP50:      {ap:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  GT count:  {n_gt}")
    print(f"  Detections:{n_dets}")

map50 = np.mean(list(aps.values()))
print(f"\nOverall mAP50: {map50:.4f}")
print("="*50)

# ── Compare against your previous single model ────────────────────────────────
print("\nComparison:")
print(f"{'Metric':<20} {'YOLOv8n (single)':>18} {'Combined':>12}")
print("-" * 52)
print(f"{'mAP50':<20} {'0.556':>18} {map50:>12.4f}")
print(f"{'Person mAP50':<20} {'0.248':>18} {aps.get(0, 0):>12.4f}")
print(f"{'Bag mAP50':<20} {'0.865':>18} {aps.get(1, 0):>12.4f}")

In [ ]:
from ultralytics import YOLO
from PIL import Image
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

coco_model = YOLO("yolov8n.pt")

val_img_dir   = "../../data/raw_data/final_unified_dataset/images/val"
val_label_dir = "../../data/raw_data/final_unified_dataset/labels/val"

# Find images where COCO detects more people than labels say
over_detected = []

for fname in os.listdir(val_img_dir)[:200]:  # sample 200
    if not fname.endswith((".jpg", ".png")):
        continue

    img_path   = os.path.join(val_img_dir, fname)
    label_path = os.path.join(val_label_dir, fname.replace(".jpg", ".txt").replace(".png", ".txt"))

    results = coco_model.predict(img_path, classes=[0], conf=0.40, verbose=False)
    coco_count = len(results[0].boxes)

    gt_count = 0
    if os.path.exists(label_path):
        with open(label_path) as f:
            gt_count = sum(1 for l in f if l.startswith("0 "))

    if coco_count > gt_count:
        over_detected.append((fname, coco_count, gt_count))

print(f"Images where COCO found more people than labelled: {len(over_detected)}")
print("\nTop cases:")
for fname, coco_n, gt_n in sorted(over_detected, key=lambda x: -(x[1]-x[2]))[:5]:
    print(f"  {fname}: COCO={coco_n}, GT={gt_n}")

In [ ]:
from ultralytics import YOLO
from PIL import Image
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches

coco_model = YOLO("yolov8n.pt")

val_img_dir   = "../../data/raw_data/final_unified_dataset/images/val"
val_label_dir = "../../data/raw_data/final_unified_dataset/labels/val"

worst_cases = [
    "3a78bd605848b134.jpg",
    "cc413a5d89c5d8e6.jpg",
    "22fa8c0824206437.jpg",
]

fig, axes = plt.subplots(1, len(worst_cases), figsize=(18, 6))

for ax, fname in zip(axes, worst_cases):
    img_path   = os.path.join(val_img_dir, fname)
    label_path = os.path.join(val_label_dir, fname.replace(".jpg", ".txt"))

    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)

    # Draw ground truth labels in green
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts or int(parts[0]) != 0:
                    continue
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                rect = patches.Rectangle(
                    (x1, y1), bw*w, bh*h,
                    linewidth=2, edgecolor="green", facecolor="none"
                )
                ax.add_patch(rect)
                ax.text(x1, y1-5, "GT person", color="green", fontsize=7)

    # Draw COCO detections in cyan
    results = coco_model.predict(img_path, classes=[0], conf=0.40, verbose=False)
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor="cyan", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x1, y2+10, f"COCO {conf:.2f}", color="cyan", fontsize=7)

    gt_count = sum(1 for l in open(label_path) if l.startswith("0 ")) \
               if os.path.exists(label_path) else 0
    ax.set_title(f"{fname[:20]}\nCOCO={len(results[0].boxes)} GT={gt_count}")
    ax.axis("off")

plt.suptitle("Green = ground truth labels | Cyan = COCO detections", fontsize=12)
plt.tight_layout()
plt.savefig("coco_vs_gt_comparison.png", dpi=100)
plt.show()


#### Webcam script for this model

In [ ]:
from ultralytics import YOLO
import cv2
import torch
import torchvision.ops as ops
import numpy as np
from collections import deque

# ── Load models ───────────────────────────────────────────────────────────────

bag_model  = YOLO("../../runs/detect/yolov8m_Bag_only_v5/train/weights/best.pt")
coco_model = YOLO("yolov8n.pt")

# ── Detection helpers ─────────────────────────────────────────────────────────

def merge_person_detections(your_boxes, coco_boxes, iou_threshold=0.5):
    if len(your_boxes) == 0 and len(coco_boxes) == 0:
        return []

    all_boxes, all_scores, all_sources = [], [], []

    for box in your_boxes:
        all_boxes.append(box.xyxy[0].cpu().tolist())
        all_scores.append(float(box.conf[0]))
        all_sources.append("yours")

    for box in coco_boxes:
        all_boxes.append(box.xyxy[0].cpu().tolist())
        all_scores.append(float(box.conf[0]))
        all_sources.append("coco")

    boxes_tensor  = torch.tensor(all_boxes,  dtype=torch.float32)
    scores_tensor = torch.tensor(all_scores, dtype=torch.float32)
    keep          = ops.nms(boxes_tensor, scores_tensor, iou_threshold)

    return [{
        "box":    all_boxes[i],
        "conf":   all_scores[i],
        "source": all_sources[i],
    } for i in keep.tolist()]


def detect_frame(frame):
    # Bags from your specialist model
    bag_results  = bag_model.predict(frame,  classes=[1], conf=0.29, verbose=False)
    # People from COCO
    coco_results = coco_model.predict(frame, classes=[0], conf=0.40, verbose=False)
    # Your model's person detections as supplement
    your_results = bag_model.predict(frame,  classes=[0], conf=0.25, verbose=False)

    your_people = list(your_results[0].boxes)
    coco_people = list(coco_results[0].boxes)
    bags        = list(bag_results[0].boxes)

    merged_people = merge_person_detections(your_people, coco_people)

    return merged_people, bags


def draw_detections(frame, people, bags, stable_bags):
    # Draw people
    for p in people:
        x1, y1, x2, y2 = map(int, p["box"])
        conf   = p["conf"]
        source = p["source"]
        color  = (255, 255, 0) if source == "yours" else (0, 255, 255)
        label  = f"person {conf:.2f} ({'M' if source == 'yours' else 'C'})"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, max(y1-8, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Draw bags — red if stable (real), orange if uncertain
    for box in bags:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf  = float(box.conf[0])
        color = (0, 0, 255) if stable_bags else (0, 165, 255)
        label = f"bag {conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, max(y1-8, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Stats overlay
    cv2.putText(frame,
                f"People: {len(people)}  Bags: {len(bags)}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Alert if stable bag detected
    if stable_bags:
        cv2.putText(frame, "!! BAG DETECTED !!",
                    (10, 70), cv2.FONT_HERSHEY_SIMPLEX,
                    1.0, (0, 0, 255), 3)

    # Legend
    cv2.putText(frame, "Yellow=your model  Cyan=COCO  Red=stable bag  Orange=uncertain",
                (10, frame.shape[0] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

    return frame

# ── Webcam loop ───────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam")
    exit()

# Temporal filter — bag must appear in 3 of last 5 frames to be considered real
bag_history = deque(maxlen=5)

print("Running... press 'q' to quit")
print("Yellow = your model person | Cyan = COCO person")
print("Red = confirmed bag | Orange = uncertain bag")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    people, bags = detect_frame(frame)

    # Update temporal filter
    bag_history.append(len(bags) > 0)
    stable_bags = sum(bag_history) >= 3  # confirmed in 3/5 frames

    annotated = draw_detections(frame.copy(), people, bags, stable_bags)

    cv2.imshow("Suspicious Bag Detection", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## Webcam detection script

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO("../../runs/detect/yolo8n_Dataset_v4/train/weights/best.pt")  # update path to your best.pt

# Open webcam
cap = cv2.VideoCapture(0)  # 0 = default webcam

if not cap.isOpened():
    print("Error: Could not open webcam")
    exit()

print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run inference on frame
    results = model.predict(
        source=frame,
        conf=0.25,
        verbose=False,
    )

    # Draw boxes on frame
    annotated = results[0].plot()

    cv2.imshow("Suspicious Bag Detection", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
from ultralytics import YOLO
import cv2
import torch
import torchvision.ops as ops
import numpy as np

# Your trained model — bags + your person detections
your_model = YOLO("../runs/detect/yolo8n_Dataset_v4/train/weights/best.pt")

# COCO pretrained — strong person detector
coco_model = YOLO("yolov8n.pt")

def merge_person_detections(your_boxes, coco_boxes, iou_threshold=0.5):
    """
    Merge person detections from both models.
    Uses NMS to remove duplicates where both detected the same person.
    Keeps the higher confidence box when there's overlap.
    """
    if len(your_boxes) == 0 and len(coco_boxes) == 0:
        return []
    
    all_boxes = []
    all_scores = []
    all_sources = []

    for box in your_boxes:
        all_boxes.append(box.xyxy[0].cpu().tolist())
        all_scores.append(float(box.conf[0]))
        all_sources.append("yours")

    for box in coco_boxes:
        all_boxes.append(box.xyxy[0].cpu().tolist())
        all_scores.append(float(box.conf[0]))
        all_sources.append("coco")

    if not all_boxes:
        return []

    boxes_tensor  = torch.tensor(all_boxes,  dtype=torch.float32)
    scores_tensor = torch.tensor(all_scores, dtype=torch.float32)

    # NMS removes overlapping boxes keeping highest confidence
    keep_indices = ops.nms(boxes_tensor, scores_tensor, iou_threshold)

    merged = []
    for i in keep_indices.tolist():
        merged.append({
            "box":    all_boxes[i],
            "conf":   all_scores[i],
            "source": all_sources[i],
        })
    return merged


def detect_frame(frame, conf_your=0.25, conf_coco=0.40):
    # Your model — get both people and bags
    your_results  = your_model.predict(frame, conf=conf_your, verbose=False)
    # COCO — get people only
    coco_results  = coco_model.predict(frame, classes=[0], conf=conf_coco, verbose=False)

    your_people = [b for b in your_results[0].boxes if int(b.cls[0]) == 0]
    your_bags   = [b for b in your_results[0].boxes if int(b.cls[0]) == 1]
    coco_people = list(coco_results[0].boxes)

    # Merge person detections from both models
    merged_people = merge_person_detections(your_people, coco_people)

    return merged_people, your_bags


def draw_detections(frame, people, bags):
    for p in people:
        x1, y1, x2, y2 = map(int, p["box"])
        conf   = p["conf"]
        source = p["source"]
        # Different shade for each source so you can see the contribution
        color  = (255, 255, 0) if source == "yours" else (0, 255, 255)
        label  = f"person {conf:.2f} ({'M' if source == 'yours' else 'C'})"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    for box in bags:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
        cv2.putText(frame, f"bag {conf:.2f}", (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 2)

    # Stats overlay
    cv2.putText(frame, f"People: {len(people)}  Bags: {len(bags)}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return frame


# ── Webcam loop ───────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam")
    exit()

print("Press 'q' to quit | M = your model (yellow) | C = COCO (cyan)")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    people, bags = detect_frame(frame)
    annotated    = draw_detections(frame, people, bags)

    cv2.imshow("Suspicious Bag Detection", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()